### Bước 1 — Copy code, vá 2 file

Vá 2 file:
1. `common.py`, hàm `read_jsonl`: tách dòng bằng `split("\n")` thay vì `splitlines()` (lỗi ký tự `\x85`).
2. `runner.py` (3 chỗ): thêm phương pháp `fixed_window_dense` vào danh sách cho phép, thêm nhánh dựng cửa sổ cố định khi cần Dense, và thêm nhánh chạy Dense trên cửa sổ cố định.

In [1]:
import glob, shutil, os

hits = glob.glob('/kaggle/input/**/requirements-research.txt', recursive=True)
assert hits, 'Không thấy requirements-research.txt: kiểm tra đã gắn dataset code chưa'
RESEARCH_SRC = os.path.dirname(hits[0])
os.makedirs('/kaggle/working/repo', exist_ok=True)
shutil.copytree(RESEARCH_SRC, '/kaggle/working/repo/research', dirs_exist_ok=True)
!pip install -q -r /kaggle/working/repo/research/requirements-research.txt

def patch(path, pairs):
    s = open(path, encoding="utf-8").read()
    for old, new in pairs:
        assert s.count(old) == 1, f"{path}: tìm thấy {s.count(old)} chỗ, cần đúng 1 :: {old[:50]}"
        s = s.replace(old, new)
    open(path, "w", encoding="utf-8").write(s)

R = "/kaggle/working/repo/research/offline_retrieval"

patch(f"{R}/common.py", [(
    'enumerate(path.read_text(encoding="utf-8").splitlines(), 1)',
    'enumerate(path.read_text(encoding="utf-8").split("\\n"), 1)')])

patch(f"{R}/runner.py", [
 ('''ALLOWED_METHODS = {
    "dense_cross_encoder",
    "fixed_window_bm25",''',
  '''ALLOWED_METHODS = {
    "dense_cross_encoder",
    "fixed_window_bm25",
    "fixed_window_dense",'''),
 ('''    fixed_windows: list[dict[str, Any]] = []
    fixed_bm25: dict[str, list[dict[str, Any]]] = {}
    if needs_fixed_bm25:
        stage_start = time.perf_counter()
        fixed_windows = build_fixed_windows(
            loaded["documents"],
            size_tokens=int(config["fixed_window"]["size_tokens"]),
            stride_tokens=int(config["fixed_window"]["stride_tokens"]),
        )
        stage_timings["fixed_window_build"] = time.perf_counter() - stage_start
        stage_start = time.perf_counter()''',
  '''    needs_fixed_dense = "fixed_window_dense" in selected_methods
    fixed_windows: list[dict[str, Any]] = []
    fixed_bm25: dict[str, list[dict[str, Any]]] = {}
    if needs_fixed_bm25 or needs_fixed_dense:
        stage_start = time.perf_counter()
        fixed_windows = build_fixed_windows(
            loaded["documents"],
            size_tokens=int(config["fixed_window"]["size_tokens"]),
            stride_tokens=int(config["fixed_window"]["stride_tokens"]),
        )
        stage_timings["fixed_window_build"] = time.perf_counter() - stage_start
    if needs_fixed_bm25:
        stage_start = time.perf_counter()'''),
 ('''    if "structure_hybrid_rrf" in selected_methods:
        stage_start = time.perf_counter()
        rrf_key =''',
  '''    if needs_fixed_dense:
        if encoder is None:
            encoder = _make_encoder(config, loaded["config_path"])
        stage_start = time.perf_counter()
        fixed_dense = rank_dense(
            fixed_windows,
            loaded["queries"],
            encoder=encoder,
            top_k=candidate_depth,
            group_by_document=target_scope == "document",
        )
        stage_timings["fixed_window_dense"] = time.perf_counter() - stage_start
        rankings["fixed_window_dense"] = {
            query_id: candidates[:depth]
            for query_id, candidates in fixed_dense.items()
        }

    if "structure_hybrid_rrf" in selected_methods:
        stage_start = time.perf_counter()
        rrf_key ='''),
])
print("Đã vá xong common.py và runner.py")

Đã vá xong common.py và runner.py


### Bước 2 — Copy pool simulated sang vùng ghi được, duyệt governance

**Vì sao phải copy:** `/kaggle/input` (nơi dataset được gắn vào) chỉ đọc, mà bước duyệt governance cần ghi vào `dataset-manifest.json`. Nên copy pool đã dựng sẵn `pan-pc11-simulated-v1` sang `/kaggle/working/data`, là vùng ghi được. Việc này chỉ copy dữ liệu đã có, không dựng lại pool.

**Pool này khác Test/Train:** đây là tập bổ sung gồm 179 parent simulated (người viết lại), 280 truy vấn, 2.000 tài liệu (161 nguồn dương, 1.839 distractor), 399.148 chunk. Nó đã loại trừ toàn bộ parent, tài liệu nghi vấn và tài liệu nguồn của 5 split trước. Chỉ có một tập nên chỉ có biến `SIM_DIR` (không có `TEST_DIR`, `TRAIN_DIR`).

**Vì sao phải duyệt governance:** runner từ chối chạy nếu `dataset-manifest.json` chưa ghi `governance = APPROVED` (lỗi "Dataset is not APPROVED"). Đây là thiết kế cố ý với dữ liệu bên thứ ba. PAN-PC-11 là corpus công khai cho nghiên cứu theo điều khoản Zenodo, nên tác giả tự duyệt và ghi tên người duyệt, ngày duyệt vào manifest để truy vết.

**Kết quả đúng:** in ra danh sách đủ 5 file của pool và dòng `governance: APPROVED`. Biến `SIM_DIR` được các bước sau dùng, nên ô này phải chạy trước khi chạy Dense.

In [3]:
import json, pathlib, os, shutil
SRC = "/kaggle/input/datasets/yenkim1/pan-pc11-simulated-v1/pan-pc11-simulated-v1"
need = ['chunks.jsonl', 'dataset-manifest.json', 'documents.jsonl', 'qrels.jsonl', 'queries.jsonl']
assert all(os.path.exists(f"{SRC}/{f}") for f in need), "Thiếu file pool"
SIM_DIR = "/kaggle/working/data/pan-pc11-simulated-v1"
shutil.copytree(SRC, SIM_DIR)
mp = pathlib.Path(SIM_DIR) / "dataset-manifest.json"
manifest = json.loads(mp.read_text(encoding="utf-8"))
manifest["governance"].update({
    "status": "APPROVED", "contains_pii": False,
    "retention": "Ephemeral Kaggle run; outputs giu lai lam bang chung thuc nghiem cho KLTN",
    "approver": "Vo Thi Kim Yen (tac gia KLTN, tu duyet du lieu cong khai PAN-PC-11 theo dieu khoan Zenodo)",
    "approved_at": "2026-11-25"})
mp.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(sorted(os.listdir(SIM_DIR)), "| governance:", manifest["governance"]["status"])

['chunks.jsonl', 'dataset-manifest.json', 'documents.jsonl', 'qrels.jsonl', 'queries.jsonl'] | governance: APPROVED


### Bước 3 — Tạo config Dense và tải mô hình E5

Dataset code trên Kaggle không có config Dense, nên tạo ngay trong notebook (nội dung giống file `pan-pc-11-dense-chunking-pilot-v1.json` trong repo, chỉ đổi `experiment_id` và bỏ trường ghi chú `revision_source`). Hai phương pháp: `fixed_window_dense` (cửa sổ cố định 256 từ, bước 128) và `structure_dense` (đoạn theo cấu trúc, tối đa 1.500 ký tự).

Cấu hình giống lần chạy Test để so sánh được: mô hình `intfloat/multilingual-e5-small` khóa đúng commit, GPU T4, fp16, batch 128, một GPU.

Mô hình chỉ tải một lần (cần bật Internet); lúc chạy dùng bản đã tải (`local_files_only`).

Kết quả đúng: in `Đã tạo config và tải mô hình`.

In [7]:
import json
from huggingface_hub import snapshot_download

CONFIG_PATH = "/kaggle/working/dense_sim_config.json"
dense_config = {
  "schema_version": 1,
  "experiment_id": "pan-pc-11-dense-chunking-simulated-v1",
  "protocol": {
    "dataset_role": "CORE_SOURCE_RETRIEVAL_DIAGNOSTIC",
    "query_unit": "annotated_suspicious_span_window",
    "target_scope": "source_document",
    "corpus_scope": "DETERMINISTIC_RESOURCE_BOUNDED_PILOT",
    "comparison": "structure_dense_vs_fixed_window_dense_matched_chunking_with_bm25",
    "full_corpus_result": False,
    "official_pan_result": False
  },
  "seed": 20260923,
  "methods": ["fixed_window_dense", "structure_dense"],
  "candidate_retrieval_depth": 200,
  "retrieval_depth": 100,
  "metric_cutoffs": [1, 5, 10, 20, 100],
  "fixed_window": {"size_tokens": 256, "stride_tokens": 128},
  "bm25": {"k1": 1.2, "b": 0.75},
  "dense": {
    "backend": "sentence_transformers",
    "model": "intfloat/multilingual-e5-small",
    "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3",
    "device": "cuda",
    "dtype": "float16",
    "batch_size": 128,
    "query_prefix": "query: ",
    "document_prefix": "passage: "
  },
  "rrf": {"k": 60},
  "metrics": {"target_scope": "document", "gold_overlap_threshold": 0.5}
}
json.dump(dense_config, open(CONFIG_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
snapshot_download("intfloat/multilingual-e5-small", revision="614241f622f53c4eeff9890bdc4f31cfecc418b3")
print("Đã tạo config và tải mô hình")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 43 files:   0%|          | 0/43 [00:00<?, ?it/s]

Đã tạo config và tải mô hình


### Bước 4a — Kiểm tra hai bước vá đã có tác dụng

In [15]:
R = "/kaggle/working/repo/research/offline_retrieval"
print("fixed_window_dense trong runner:", "fixed_window_dense" in open(f"{R}/runner.py", encoding="utf-8").read())
print("dtype trong dense.py:", '"dtype"' in open(f"{R}/dense.py", encoding="utf-8").read())

fixed_window_dense trong runner: True
dtype trong dense.py: True


### Bước 4b — Chạy Dense (2 cách chunking) trên pool simulated

Chạy `fixed_window_dense` và `structure_dense` trên pool simulated (179 parent, 280 truy vấn, 399.148 chunk). Mỗi truy vấn và mỗi đoạn được mã hóa thành vector 384 chiều, độ giống đo bằng cosine. Điểm của tài liệu là điểm cao nhất trong các đoạn của nó, rồi xếp hạng tài liệu và giữ top 100. Cách gộp theo tài liệu giống hệt BM25 nên so sánh được trực tiếp.

Phần mã hóa chunk chiếm gần hết thời gian. Lần chạy này mất **58,3 phút** trên một T4 fp16: `fixed_window_dense` 37,0 phút (2.221,5 giây), `structure_dense` 19,2 phút (1.152,8 giây), dựng cửa sổ cố định 52,9 giây. Con số này gần với lần chạy Test (khoảng 60 phút), vì pool có 399.148 chunk structure, nhiều hơn Test (375.876).

Kết quả ghi vào `/kaggle/working/results/simulated/dense_chunking/`. Kết quả đúng: `rc=0`.

In [16]:
import subprocess, pathlib, time

REPO = "/kaggle/working/repo"
out_dir = "/kaggle/working/results/simulated/dense_chunking"
pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)
cmd = ["python", "-m", "research.offline_retrieval", "run",
       "--dataset-manifest", f"{SIM_DIR}/dataset-manifest.json",
       "--config", CONFIG_PATH, "--output-dir", out_dir]
t0 = time.perf_counter()
r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
print(r.stdout[-1500:])
if r.returncode != 0:
    print("LỖI:", r.stderr[-3000:])
print(f"rc={r.returncode} | {(time.perf_counter()-t0)/60:.1f} phút")

/kaggle/working/results/simulated/dense_chunking/run_manifest.json

rc=0 | 58.3 phút


### Bước 5 — Đọc Recall Dense và nén kết quả

In Recall@1, 5, 10, 20, 100 của hai phương pháp ở hai mức: `parent` (179 parent, số báo cáo chính) và `truy vấn` (280 truy vấn, chỉ tham khảo). Sau đó in thiết bị và độ chính xác đã dùng (lấy từ `run_manifest.json`) và nén thư mục `results` thành `dense_simulated_results.zip`. Tải file này về và lưu vào Drive `KLTN/simulated_extra/`.

In [17]:
import json, shutil
import pandas as pd

out_dir = "/kaggle/working/results/simulated/dense_chunking"
m = json.load(open(f"{out_dir}/metrics.json"))
rows = []
for method, v in m.items():
    if isinstance(v, dict) and "parent_macro" in v:
        for scope, label in (("parent_macro", "parent"), ("macro", "truy vấn")):
            r = {"Phương pháp": method, "Mức": label}
            r.update({f"R@{k}": round(v[scope][k]["recall"], 3) for k in ["1", "5", "10", "20", "100"]})
            rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

rm = json.load(open(f"{out_dir}/run_manifest.json"))
print("timing_seconds:", {k: round(x, 1) for k, x in rm.get("timing_seconds", {}).items()})
enc = rm.get("dense_encoder", {})
print("devices:", enc.get("devices") or enc.get("device"), "| dtype:", enc.get("dtype"))

shutil.make_archive("/kaggle/working/dense_simulated_results", "zip", "/kaggle/working/results")
print("Đã nén: /kaggle/working/dense_simulated_results.zip")

       Phương pháp      Mức   R@1   R@5  R@10  R@20  R@100
fixed_window_dense   parent 0.862 0.903 0.933 0.961  0.978
fixed_window_dense truy vấn 0.907 0.936 0.957 0.975  0.986
   structure_dense   parent 0.855 0.935 0.942 0.944  0.994
   structure_dense truy vấn 0.900 0.954 0.961 0.964  0.996
timing_seconds: {'evaluation': 0.0, 'fixed_window_build': 52.9, 'fixed_window_dense': 2221.5, 'structure_dense': 1152.8}
devices: cuda | dtype: float16
Đã nén: /kaggle/working/dense_simulated_results.zip


### Kết quả Dense trên pool `pan-pc11-simulated-v1`

Pool: 179 parent (simulated), 280 truy vấn, 2.000 tài liệu (161 nguồn dương, 1.839 distractor), 399.148 chunk. Mô hình `intfloat/multilingual-e5-small` (khóa đúng commit), T4, fp16, cùng cấu hình với lần chạy Test. Thời gian chạy: 58,3 phút.

| Phương pháp | Mức | R@1 | R@5 | R@10 | R@20 | R@100 |
|---|---|---:|---:|---:|---:|---:|
| fixed_window_dense | parent | 0,862 | 0,903 | 0,933 | 0,961 | 0,978 |
| structure_dense | parent | 0,855 | 0,935 | 0,942 | 0,944 | 0,994 |
| fixed_window_dense | truy vấn | 0,907 | 0,936 | 0,957 | 0,975 | 0,986 |
| structure_dense | truy vấn | 0,900 | 0,954 | 0,961 | 0,964 | 0,996 |

**Cách đọc:**
- Số mức **parent** (179) là số báo cáo chính; mức truy vấn (280) chỉ để tham khảo.
- Dense yếu hơn BM25 ở R@1 trên tập này (0,862 so với 0,933 ở cửa sổ cố định, cùng mức parent).
- Hai cách chunking không cùng chiều ở mọi k: cửa sổ cố định hơn ở R@1 và R@20; structure hơn ở R@5, R@10 và R@100 (0,994 so với 0,978). Với 179 parent, một parent đổi khoảng 0,6 điểm phần trăm, nên chưa kết luận về hai cách chunking từ bảng này.
- Bước 6b của notebook Colab dùng `fixed_window_dense` (R@1 = 0,862) để dựng danh sách K = 50.

Kết quả ghi tại `/kaggle/working/results/simulated/dense_chunking/` (`predictions.jsonl`, `metrics.json`, `run_manifest.json`), nén thành `dense_simulated_results.zip`.